# Teaching Agents to Ask for Help
## GRPO Training on Agent World Model with TRL + OpenEnv

This notebook trains a small LLM (Qwen3-0.6B) to use MCP tools via reinforcement learning.
The agent learns to interact with 1,000 simulated web environments — each backed by a SQLite DB and REST API.

**Key idea:** The agent has access to a GPT expert it can call when stuck. We use reward shaping to teach it
*when* to ask for help vs solve tasks independently.

| Reward | Pattern |
|--------|--------|
| **2.0** | Solo completion (no expert) |
| **1.3** | Recovery expert (tried first, then asked) |
| **0.8** | Blind expert (asked immediately) |
| **-1.0** | Format error (malformed XML) |

For full results and analysis, see [EXPERT_ENHANCEMENT.md](https://github.com/sfc-gh-mhidayetoglu/OpenEnv/blob/add-agent-world-model/envs/agent_world_model_env/EXPERT_ENHANCEMENT.md)

## 1. Setup

Install dependencies and clone the environment.

In [ ]:
# Install TRL with vLLM support, OpenEnv core, and other deps
!pip install -q "trl[vllm]" "openenv-core[core]>=0.2.1" openai datasets huggingface-hub

# Clone the OpenEnv repo and install the AWM environment
!git clone --branch add-agent-world-model https://github.com/sfc-gh-mhidayetoglu/OpenEnv.git /tmp/OpenEnv 2>/dev/null || true
!cd /tmp/OpenEnv/envs/agent_world_model_env && pip install -q -e .

## 2. Configure API Keys

Set your Azure OpenAI (or OpenAI) credentials for the expert tool.
If you don't have credentials, the training will still work — just without the expert.

In [ ]:
import os

# Option A: Azure OpenAI (for expert tool)
# os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-endpoint.openai.azure.com/"
# os.environ["AZURE_OPENAI_API_KEY"] = "your-key"
# os.environ["EXPERT_MODEL"] = "gpt-4o"

# Option B: OpenAI API
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["EXPERT_MODEL"] = "gpt-4o"

USE_EXPERT = "AZURE_OPENAI_API_KEY" in os.environ or "OPENAI_API_KEY" in os.environ
print(f"Expert tool: {'enabled' if USE_EXPERT else 'disabled (no API key)'}")

## 3. Start the AWM Environment Server

The server hosts 1,000 web environments as MCP tool servers. Each environment has:
- ~35 API tools (CRUD endpoints over a SQLite database)
- 10 tasks (natural language instructions)
- A code verifier that checks the final DB state

In [ ]:
import subprocess, time, requests

ENV_PORT = 8899
ENV_URL = f"http://localhost:{ENV_PORT}"

# Start the AWM server in the background
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "agent_world_model_env.server.app:app",
     "--host", "127.0.0.1", "--port", str(ENV_PORT)],
    cwd="/tmp/OpenEnv/envs/agent_world_model_env",
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)

# Wait for server to be ready
for i in range(30):
    try:
        r = requests.get(f"{ENV_URL}/health", timeout=2)
        if r.status_code == 200:
            print(f"AWM server ready on port {ENV_PORT}")
            break
    except:
        pass
    time.sleep(2)
    print(f"Waiting for server... ({i+1}/30)")
else:
    print("Server didn't start. Check stderr:")
    print(server_proc.stderr.read().decode()[:2000])

## 4. Quick Test: Run One Episode

Let's verify the environment works by running a single task.

In [ ]:
import asyncio, json
from agent_world_model_env import AWMEnv
from openenv.core.env_server.mcp_types import CallToolAction, ListToolsAction

async def test_env():
    async with AWMEnv(base_url=ENV_URL) as env:
        # List available scenarios
        result = await env.step(CallToolAction(tool_name="__list_scenarios__", arguments={}))
        scenarios = result.observation.scenarios
        print(f"Total environments: {result.observation.total}")
        print(f"\nSample environments:")
        for s in scenarios[:5]:
            print(f"  - {s['name']}: {s['num_tasks']} tasks")
            print(f"    Example: {s['tasks'][0][:80]}...")

        # Reset to a specific task
        result = await env.reset(scenario="flowlatch_automation_studio", task_idx=0)
        print(f"\nTask: {result.observation.task[:200]}")
        print(f"Tools available: {result.observation.num_tools}")

        # List tools
        tools = await env.step(ListToolsAction())
        print(f"\nFirst 5 tools:")
        for t in tools.observation.tools[:5]:
            print(f"  - {t.name}: {t.description[:60]}")

        # Cleanup
        await env.step(CallToolAction(tool_name="done", arguments={}))

await test_env()

## 5. Prepare Training Data

We create a dataset of prompts from workflow automation scenarios.
Each prompt contains the task description and instructions for using tools.

In [ ]:
import sys
sys.path.insert(0, "/tmp/OpenEnv/envs/agent_world_model_env")

from agent_world_model_env.server.data_loader import AWMDataLoader
from datasets import Dataset

SYSTEM_PROMPT = """You are an MCP tool-use agent. You interact with an environment via tool calls.
At each step, call exactly ONE tool using XML tags:

<tool_call>
{"name": "call_tool", "arguments": {"tool_name": "TOOL_NAME", "arguments": "{\\"param\\": \\"value\\"}"}}
</tool_call>

Available meta-tools:
1. list_tools - discover environment tools
2. call_tool - call a specific tool

Start by calling list_tools, then use call_tool to complete the task.
When done, output your final answer as plain text."""

# Load tasks from AWM dataset
loader = AWMDataLoader()

# Use a few workflow automation scenarios for training
TRAIN_SCENARIOS = [
    "flowlatch_automation_studio",
    "flowmesh_workflow_automation_hub",
    "flowroute_approval_automation",
    "flowtrack_team_workflow_hub",
]

scenarios = []
prompts = []
for scenario_name in TRAIN_SCENARIOS:
    tasks = loader.get_tasks(scenario_name)
    for i, task_text in enumerate(tasks[:3]):  # 3 tasks per scenario = 12 total
        scenarios.append({"scenario": scenario_name, "task_idx": i, "task": task_text})
        prompt = f"{SYSTEM_PROMPT}\n\nTask: {task_text}\n\nStart by calling list_tools."
        prompts.append(prompt)

dataset = Dataset.from_dict({"prompt": prompts})
print(f"Training dataset: {len(dataset)} prompts from {len(TRAIN_SCENARIOS)} environments")
print(f"\nSample task:")
print(scenarios[0]["task"][:200])

## 6. Define Rollout and Reward Functions

The rollout function generates completions, runs them in the environment, and collects rewards.
Reward functions extract signals from the rollout results.

In [ ]:
import re, asyncio
from trl import GRPOTrainer
from trl.experimental.openenv import generate_rollout_completions


def parse_tool_call(text):
    """Extract <tool_call> block from model output."""
    m = re.search(r"<tool_call>\s*(.*?)\s*</tool_call>", text, re.DOTALL)
    if not m:
        return None
    try:
        data = json.loads(m.group(1).strip())
    except json.JSONDecodeError:
        return None
    if isinstance(data, list):
        data = data[0] if data else None
    if not isinstance(data, dict) or "name" not in data:
        return None
    return data


async def run_episode_async(scenario, task_idx, completion_text):
    """Run one episode: execute the model's tool calls in the environment."""
    try:
        async with AWMEnv(base_url=ENV_URL) as env:
            await env.reset(scenario=scenario, task_idx=task_idx)

            # Try to execute the first tool call from the completion
            tc = parse_tool_call(completion_text)
            if tc and tc["name"] == "call_tool":
                args = tc.get("arguments", {})
                tool_name = args.get("tool_name", "")
                inner_args = args.get("arguments", "{}")
                if isinstance(inner_args, str):
                    try:
                        inner_args = json.loads(inner_args)
                    except:
                        inner_args = {}
                await env.step(CallToolAction(tool_name=tool_name, arguments=inner_args or {}))
            elif tc and tc["name"] == "list_tools":
                await env.step(ListToolsAction())

            # Verify
            result = await env.step(CallToolAction(
                tool_name="verify",
                arguments={"verifier_mode": "code", "final_answer": completion_text},
            ))
            reward = float(result.reward or 0.0)
            classification = getattr(result.observation, "reward_type", "unknown")
            await env.step(CallToolAction(tool_name="done", arguments={}))
            return reward, classification
    except Exception as e:
        return -1.0, "error"


def rollout_func(input_prompts, trainer):
    """Custom rollout: generate completions, run in env, collect rewards."""
    outputs = generate_rollout_completions(trainer, input_prompts)
    tokenizer = trainer.processing_class

    completions_text = [
        tokenizer.decode(out["completion_ids"], skip_special_tokens=True)
        for out in outputs
    ]

    # Run each completion in the environment
    loop = asyncio.new_event_loop()
    env_rewards = []
    classifications = []

    for i, text in enumerate(completions_text):
        s = scenarios[i % len(scenarios)]
        reward, cls = loop.run_until_complete(
            run_episode_async(s["scenario"], s["task_idx"], text)
        )
        env_rewards.append(reward)
        classifications.append(cls)
    loop.close()

    return {
        "prompt_ids": [out["prompt_ids"] for out in outputs],
        "completion_ids": [out["completion_ids"] for out in outputs],
        "logprobs": [out["logprobs"] for out in outputs],
        "env_reward": env_rewards,
        "classification": classifications,
    }


# Reward functions
def reward_from_env(completions, **kwargs):
    """Environment reward: 1.0 for complete, -1.0 for format error."""
    rewards = kwargs.get("env_reward", [])
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)


def reward_format(completions, **kwargs):
    """Small bonus for producing valid XML tool calls."""
    rewards = []
    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else str(c)
        if parse_tool_call(text) is not None:
            rewards.append(0.1)  # valid format
        elif "<tool_call>" in text:
            rewards.append(-0.3)  # tried but malformed
        else:
            rewards.append(0.0)  # no attempt
    return rewards


print("Rollout and reward functions defined.")

## 7. Train!

This runs GRPO training using:
- **Model:** Qwen3-0.6B (small enough for Colab T4/A100)
- **Environment:** AWM workflow automation tasks
- **Reward:** Environment verifier + format bonus
- **vLLM:** Colocated mode (single GPU)

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="./output_awm_grpo",
    use_vllm=True,
    vllm_mode="colocate",
    num_generations=4,           # 4 completions per prompt
    max_completion_length=2048,  # enough for multi-step tool use
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_steps=20,                # quick demo; increase for real training
    learning_rate=5e-6,
    temperature=1.0,
    beta=0.001,                  # small KL penalty
    logging_steps=1,
    save_steps=10,
    report_to="none",
)

trainer = GRPOTrainer(
    model="Qwen/Qwen3-0.6B",
    reward_funcs=[reward_from_env, reward_format],
    train_dataset=dataset,
    rollout_func=rollout_func,
    args=grpo_config,
)

print("Starting GRPO training on Agent World Model...")
print(f"  Model: Qwen/Qwen3-0.6B")
print(f"  Tasks: {len(dataset)} prompts x {grpo_config.num_generations} generations = {len(dataset) * grpo_config.num_generations} rollouts/epoch")
print(f"  Steps: {grpo_config.max_steps}")
print(f"  Expert: {'enabled' if USE_EXPERT else 'disabled'}")
print()

trainer.train()

## 8. Evaluate

Test the trained model on a held-out task.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./output_awm_grpo"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")

# Test on a held-out scenario
test_task = "Create a new workflow named 'Test Automation' in draft status with a trigger on record_created."
test_prompt = f"{SYSTEM_PROMPT}\n\nTask: {test_task}\n\nStart by calling list_tools."

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)

completion = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("Model output:")
print(completion[:500])

tc = parse_tool_call(completion)
if tc:
    print(f"\nParsed tool call: {json.dumps(tc, indent=2)}")
else:
    print("\nNo valid tool call found in output.")

## 9. Cleanup

In [ ]:
server_proc.terminate()
print("AWM server stopped.")

---

## What's Next?

This minimal notebook demonstrates the core training loop. Our full training setup achieves:

| Metric | Step 1 | Step 29 |
|--------|--------|---------|
| Completion rate | 5.5% | **44.5%** |
| Reward | -0.528 | **+0.824 peak** |
| Format errors | 88% | **31%** |

To reproduce those results, scale up:
- **Model:** Qwen3-4B (instead of 0.6B)
- **GPUs:** 8x B200 with distributed training (verl/ray)
- **Tasks:** 53 training + 29 validation scenarios
- **Expert:** GPT-5.1 with verifier-informed prompts
- **Adaptive ratio:** Scaffold (6E/2S) -> Balanced (4E/4S) -> Independence (2E/6S)
- **Reward shaping:** Solo bonus (+1.0), Recovery bonus (+0.3), Blind penalty (-0.2)

See the full analysis: [EXPERT_ENHANCEMENT.md](https://github.com/sfc-gh-mhidayetoglu/OpenEnv/blob/add-agent-world-model/envs/agent_world_model_env/EXPERT_ENHANCEMENT.md)

### References
- [Agent World Model paper](https://arxiv.org/abs/2602.10090)
- [AgentWorldModel-1K dataset](https://huggingface.co/datasets/Snowflake/AgentWorldModel-1K)
- [TRL OpenEnv docs](https://huggingface.co/docs/trl/main/openenv)
- [OpenEnv framework](https://github.com/meta-pytorch/OpenEnv)